# B — Compréhension des modèles (barème validation)

**Rôle** : document **critère B** pour **tous** les algorithmes utilisés dans **01_E → 04_F** — intuition, **paramètres** typiques, **hypothèses**, **limites**, et **pourquoi** chaque paire de modèles correspond à l’**objectif** / à la **cible** de la tâche.

Un correcteur attend souvent trois niveaux de lecture : (1) **ce que fait** l’algorithme en une phrase, (2) **à quelles conditions** il est fiable, (3) **ce qu’il ne peut pas** capturer. Les sections suivantes couvrent ces trois niveaux pour chaque paire, en lien direct avec les notebooks d’entraînement.

**Pas d’entraînement ici** : exécuter **00_A** (données) puis **01_E … 04_F** ; **05** agrège les métriques. Ce notebook sert au **rapport** et à la **relecture barème**.

| Notebook | Tâche | Cible / objectif | Deux modèles comparés |
|----------|-------|------------------|------------------------|
| **01_E** | Clustering | Segments (pas de label vrai) | **K-Means** vs **Agglomératif (Ward)** |
| **02_C** | Classification | `reservation_status` | **Random Forest** vs **Régression logistique** (multinomiale) |
| **03_D** | Régression | 1er disponible parmi `final_price`, `service_price`, … | **Ridge** vs **Forêt aléatoire (régresseur)** |
| **04_F** | Séries | Série mensuelle (ex. volume agrégé) | **Holt** (lissage exp. + tendance) vs **ARIMA** |

Les **métriques chiffrées** et le **champion** par tâche sont dans **05** et les `metrics_*.json`. Ici on documente le **sens** des algorithmes et le **lien direct** entre type de problème (segmentation, classe, montant, temps) et choix des deux approches comparées.


## E — Clustering (`01_E_clustering_segmentation.ipynb`)

**Objectif métier** : typer des entités / lignes (performance wide ou RFM) pour **piloter l’offre** et la diversité de segments — **pas de variable cible supervisée** ; le « bon » modèle se juge par **cohésion / séparation** (silhouette, Davies-Bouldin, coude).

En segmentation, le risque principal est **sur-interpréter** un cluster : les étiquettes 0,1, 2… n’ont pas de sens intrinsèque. Les **heatmaps de profils** et la **PCA** servent à traduire chaque groupe en langage métier (« fort panier / faible fréquence », etc.), toujours sous réserve de la qualité des scores de validation.

### Modèle 1 — K-Means

| | |
|---|---|
| **Intuition** | Partitionner l’espace en **k** groupes en minimisant la variance **intra**-cluster (distance au centroïde). |
| **Paramètres clés** | `n_clusters` (k), `n_init`, `random_state` — dans le notebook, k issu du **coude** / validation. |
| **Hypothèses / idéal** | Clusters **compacts**, souvent **convexes** ; distance euclidienne pertinente après **standardisation**. |
| **Limites** | Sensible au **choix de k** et aux **échelles** (d’où le scaler) ; formes allongées ou densités variables : sous-optimal. |
| **Lien objectif** | Rapide, centroïdes **interprétables** (heatmap) → **nommer** les segments en BI. |

### Modèle 2 — Clustering hiérarchique (Ward)

| | |
|---|---|
| **Intuition** | Fusionner progressivement les points en minimisant la **variance intra** à chaque fusion (lien **Ward**). |
| **Paramètres clés** | `n_clusters`, `linkage='ward'` — même k que K-Means pour comparaison **équitable**. |
| **Hypothèses** | Hiérarchie **nested** ; Ward favorise des groupes de **taille / variance** comparable. |
| **Limites** | Coût plus élevé sur très grands *n* ; pas de centroïdes simples comme K-Means. |
| **Lien objectif** | Offre une **alternative** sans centroïdes iteratifs ; utile si la structure est **hiérarchique** plutôt que « boules » sphériques. |

**Pourquoi ces deux ?** Exiger **≥ 2** méthodes **non supervisées** différentes (partition vs hiérarchie) pour ne pas baser la segmentation sur **une seule** hypothèse géométrique.

Si les deux modèles donnent des profils **très proches** pour un même *k*, la segmentation est **robuste** à la méthode ; s’ils divergent fortement, c’est un signal pour **re-voir k**, les variables d’entrée, ou la pertinence du jeu wide vs agrégats RFM.


## C — Classification (`02_C_classification_statut_reservation.ipynb`)

**Cible** : **`reservation_status`** (classes issues du DW — funnel / statuts de réservation). **Objectif** : anticiper le statut à partir de features numériques pour **KPI funnel** (acceptation, annulation, etc.).

Le **déséquilibre** des classes est typique des funnels : sans `class_weight='balanced'`, le modèle peut « exceller » en accuracy en ignorant les statuts rares mais critiques. Les métriques **F1 pondéré** et la **matrice de confusion** donnent une lecture plus honnête que l’accuracy seule.

### Modèle 1 — Random Forest (classificateur)

| | |
|---|---|
| **Intuition** | Ensemble d’**arbres** ; vote majoritaire ; capture des **seuils** et interactions **non linéaires**. |
| **Paramètres clés** | `n_estimators`, `max_depth` (GridSearch) ; `class_weight='balanced'` pour le **déséquilibre** des classes. |
| **Hypothèses** | Peu de formalisme ; besoin de **représentativité** et de **tuning** pour limiter le sur-apprentissage. |
| **Limites** | Moins **transparent** que la régression logistique ; peut **sur-apprendre** si arbres trop profonds. |
| **Lien cible** | Adapté aux relations **non linéaires** entre signaux financiers / calendrier et **statut** ; **importances** pour l’explicabilité métier. |

### Modèle 2 — Régression logistique (multinomiale)

| | |
|---|---|
| **Intuition** | Modèle **linéaire** sur les **log-odds** de chaque classe ; frontières de décision liées aux **scores** des features. |
| **Paramètres clés** | `C` (régularisation L2, GridSearch) ; **StandardScaler** dans le pipeline pour comparer les **coefficients**. |
| **Hypothèses** | Séparabilité **approximativement linéaire** après transformation ; **multiclasse** gérée (multinomiale). |
| **Limites** | Peut **sous-performer** si les frontières sont très non linéaires ; sensible aux **corrélations fortes** entre features. |
| **Lien cible** | **Baseline forte** et **interprétable** (coefficients) pour expliquer le funnel aux parties prenantes. |

**Pourquoi ces deux ?** **Non-linéaire** (RF) vs **linéaire interprétable** (LR) ; les deux avec **`class_weight='balanced'`** car la cible métier est souvent **déséquilibrée** — aligné sur **Accuracy, F1, ROC-AUC** du barème **C**.

En cas de scores proches sur le test, le choix peut se faire sur la **lisibilité** (LR pour expliquer le funnel) ou la **robustesse aux interactions** (RF) — à expliciter dans le rapport selon le coût des erreurs par statut.


## D — Régression (`03_D_regression_montants_KPI.ipynb`)

**Cible** : **première variable continue disponible** parmi `final_price`, `service_price`, `benchmark_avg_price`, `event_budget`, `commission_margin`. **Objectif** : estimer un **montant / KPI continu** pour le pilotage financier (panier, budget, marge, etc.).

La régression exige de surveiller **fuites** possibles (cible dupliquée dans `X`, doublons) et la **forme des résidus** : un R² élevé sans nuage réel/prédit crédible doit être interrogé avant toute conclusion métier.

### Modèle 1 — Ridge

| | |
|---|---|
| **Intuition** | Régression **linéaire** avec pénalité **L2** sur les coefficients — réduit la variance quand les features sont corrélées. |
| **Paramètres clés** | `alpha` (force de régularisation) ; pipeline avec **StandardScaler**. |
| **Hypothèses** | Relation **approximativement linéaire** entre X et y ; résidus souvent analysés (homoscédasticité idéale en théorie). |
| **Limites** | Ne capte pas les **non-linéarités** fortes ni les **seuils** ; outliers peuvent tirer les coefficients (robustesse modérée). |
| **Lien cible** | **Coefficients** lisibles après mise à l’échelle → **explicabilité** des leviers sur le **montant**. |

### Modèle 2 — Random Forest (régresseur)

| | |
|---|---|
| **Intuition** | Moyenne d’arbres qui partitionnent l’espace des features — modèle **non paramétrique** « par morceaux ». |
| **Paramètres clés** | `n_estimators`, profondeur (fixés dans le notebook) ; scaling en amont pour cohérence pipeline. |
| **Hypothèses** | Peu restrictives sur la forme de la relation ; besoin de **CV** pour estimer la généralisation. |
| **Limites** | Moins trivial à **expliquer** globalement que Ridge ; attention au **sur-apprentissage** si arbres trop profonds. |
| **Lien cible** | Mieux adapté si le **prix / montant** dépend de **interactions** et seuils (offre, date, catégorie encodée). |

**Pourquoi ces deux ?** Couvrir **linéaire régularisé** (simple, coefs) vs **non-linéaire** (flexible, importances) — champion choisi par **RMSE test** (et **MSE, MAE, R²** du barème **D**).

Le **RMSE** pénalise surtout les grosses erreurs ; le **MAE** décrit l’erreur typique en unités de la cible. Si votre métier redoute les outliers (gros montants mal prédits), le RMSE est souvent le critère de discussion principal.


## F — Séries temporelles (`04_F_series_temporelles.ipynb`)

**Cible** : **série mensuelle** agrégée (ex. volume d’activité `nb_fact_rows` ou autre colonne retenue). **Objectif** : **prévoir** l’évolution court terme pour **anticipation opérationnelle** (charge, tendance).

La prévision série impose un **découpage temporel** : toute validation « aléatoire » mélangerait passé et futur et surestimerait la qualité. Les tests ADF/KPSS et la décomposition aident à formuler ce que le modèle peut raisonnablement capter avec les données disponibles.

### Modèle 1 — Holt (lissage exponentiel double)

| | |
|---|---|
| **Intuition** | **Niveau** + **tendance** mise à jour par des équations de **lissage** (pas de saisonnalité dans la spec utilisée : `seasonal=None`). |
| **Paramètres clés** | Constantes de lissage estimées (`fit`) ; `trend='add'`. |
| **Hypothèses** | Tendance **localement lisse** ; pas de saisonnalité forte modélisée dans cette variante. |
| **Limites** | Si **saisonnalité mensuelle forte**, un **SARIMA** ou **Holt-Winters** pourrait mieux convenir. |
| **Lien cible** | Modèle **parcimonieux** pour séries avec **tendance** — souvent **robuste** et **simple** à expliquer aux métiers. |

### Modèle 2 — ARIMA

| | |
|---|---|
| **Intuition** | **AR** (auto-régression) + **I** (différenciation pour stationnariser) + **MA** (moyenne mobile sur les erreurs). |
| **Paramètres clés** | Ordre `(p,d,q)` — dans le notebook typiquement **(1,1,1)** ou repli **(0,1,1)**. |
| **Hypothèses** | Après différenciation, résidus proches d’un **bruit blanc** ; structure markovienne du **linéaire** en l’historique. |
| **Limites** | **Linéaire** ; sensible à la **spec** d’ordre ; saisonnalité non captée si non incluse (SARIMA). |
| **Lien cible** | Standard **académique** pour séries macro ; comparer à Holt sur **RMSE / MAE / MAPE** **holdout** (barème **F**). |

**Pourquoi ces deux ?** **Lissage structuré** (Holt) vs **modèle ARMA sur différences** (ARIMA) — tous deux classiques pour **prévision univariée** après analyse **ADF / KPSS** et **décomposition** (rappelées dans **04_F**).

Le **MAPE** est lisible pour le métier (erreur en %), mais peut être instable si la série touche des valeurs très faibles ; croisez-le toujours avec **MAE** et la courbe **réel vs prédit** sur le holdout.


## Synthèse — justification des paires (lien problème → modèles)

| Critère | Problème | Pourquoi modèle A | Pourquoi modèle B |
|--------|----------|-------------------|-------------------|
| **E** | Groupes **sans** étiquette | K-Means : partitions + centroïdes | Ward : hiérarchie, autre hypothèse de fusion |
| **C** | Classe **discrète** (statut) | RF : non-linéaire, importances | LR : baseline, coefs, même pondération déséquilibre |
| **D** | **Montant** continu | Ridge : linéaire, explicable | RF : interactions / seuils sur le prix |
| **F** | **Indice temporel** | Holt : tendance, simple | ARIMA : structure AR/I/MA standard |

Dans chaque notebook **01–04**, le **champion** n’annule pas l’intérêt pédagogique du **second** modèle : il sert de **contre-modèle** pour discuter **biais-variance**, **hypothèses** et **limites** (critère **B**).

Pour la **soutenance** ou le rapport, une formulation utile est : *problème → hypothèse du modèle A → hypothèse du modèle B → ce que montrent les métriques/figures sur lequel ces hypothèses semblent tenir*.


## Note

Aucune cellule de code obligatoire : ce document complète les sections **B** déjà développées dans les notebooks **01_E, 02_C, 03_D, 04_F** (rappels locaux + figures). Pour les **métriques chiffrées**, le champion par tâche et le tableau agrégé, exécuter **05_synthese_metriques_validation.ipynb** et consulter les fichiers `metrics_*.json`.
